# VDCNN — Very Deep Convolutional Neural Network
## SQL Injection Detection Model

**Based on:** Conneau et al. 2017 — ["Very Deep Convolutional Networks for Text Classification"](https://arxiv.org/abs/1606.01781)

### Architecture Overview
```
Input (batch, 200) → Embedding(97, 16) → Conv1d(16→64)
  → Stage 1: ConvBlock(64)  + MaxPool
  → Stage 2: ConvBlock(128) + MaxPool  
  → Stage 3: ConvBlock(256) + MaxPool
  → Stage 4: ConvBlock(512)
  → KMaxPool(k=8) → FC(4096→2048→2048→1) → Sigmoid
```

**Key features:**
- Residual connections (identity + 1x1 conv projection)
- Progressive filter expansion: 64 → 128 → 256 → 512
- K-max pooling for position-invariant features
- Kaiming (He) weight initialization
- Scalable depth: 9 / 17 / 29 / 49 conv layers

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path

# Ensure project root is in path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Model Architecture

In [ ]:
from models.char_cnn_model import CharCNN, ConvBlock, KMaxPool1d, DEPTH_CONFIG, STAGE_FILTERS

# Create VDCNN-9 model
model = CharCNN(vocab_size=97, embed_dim=16, depth=9)
print(repr(model))
print(f"\nTotal trainable parameters: {model.count_parameters():,}")
print(f"\nDepth configurations available:")
for depth, blocks in DEPTH_CONFIG.items():
    m = CharCNN(vocab_size=97, embed_dim=16, depth=depth)
    print(f"  VDCNN-{depth}: blocks={blocks}, params={m.count_parameters():,}")

In [ ]:
# Detailed layer breakdown
print("=" * 70)
print("VDCNN-9 Layer-by-Layer Architecture")
print("=" * 70)
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv1d, nn.Linear, nn.MaxPool1d, nn.Embedding)):
        params = sum(p.numel() for p in module.parameters())
        print(f"{name:40s} | {str(module):45s} | params: {params:>10,}")

In [ ]:
# Test forward pass
print("Forward Pass Test")
print("-" * 50)

batch_size = 8
seq_length = 200
dummy_input = torch.randint(0, 97, (batch_size, seq_length))

model.eval()
with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}  (batch={batch_size}, seq_len={seq_length})")
print(f"Output shape: {output.shape}  (batch={batch_size}, 1)")
print(f"Output range: [{output.min().item():.4f}, {output.max().item():.4f}]")
print(f"Output values: {output.squeeze().tolist()}")
print("\n✓ Forward pass successful!")

## 2. Character Tokenizer

In [ ]:
from models.char_tokenizer import CharTokenizer

tokenizer = CharTokenizer(max_length=200)

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Max length: {tokenizer.max_length}")
print(f"PAD index: {tokenizer.pad_idx}")
print(f"UNK index: {tokenizer.unk_idx}")
print()

# Demonstrate tokenization
examples = [
    "' OR '1'='1",
    "John O'Brien",
    "'; DROP TABLE users--",
    "normal search query",
    "' UNION SELECT password FROM users--",
]

print(f"{'Input':<45} | {'Tokens (first 30)'}")
print("-" * 90)
for text in examples:
    tokens = tokenizer.encode(text)
    print(f"{text:<45} | {tokens[:30].tolist()}...")

## 3. Load and Explore Dataset

In [ ]:
# Load the training dataset
dataset_path = PROJECT_ROOT / 'SQL_Dataset_Extended.csv'

if dataset_path.exists():
    df = pd.read_csv(dataset_path)
    print(f"Dataset loaded: {len(df):,} samples")
    print(f"Columns: {list(df.columns)}")
    print(f"\nLabel distribution:")
    if 'label' in df.columns:
        label_col = 'label'
    elif 'Label' in df.columns:
        label_col = 'Label'
    else:
        label_col = df.columns[-1]  # assume last column
    
    print(df[label_col].value_counts())
    print(f"\nBalance: {df[label_col].value_counts(normalize=True).to_dict()}")
else:
    # Try alternative paths
    alt_paths = list(PROJECT_ROOT.glob('*.csv')) + list(PROJECT_ROOT.glob('data/*.csv'))
    print(f"Dataset not found at {dataset_path}")
    print(f"Available CSV files: {[str(p) for p in alt_paths]}")

In [ ]:
# Show sample data
if 'df' in dir() and df is not None:
    text_col = df.columns[0]  # first column is usually text
    
    print("=" * 80)
    print("INJECTION SAMPLES (label=1)")
    print("=" * 80)
    injections = df[df[label_col] == 1].head(10)
    for i, row in injections.iterrows():
        text = str(row[text_col])[:80]
        print(f"  [{i:5d}] {text}")
    
    print(f"\n{'=' * 80}")
    print("SAFE SAMPLES (label=0)")
    print("=" * 80)
    safe = df[df[label_col] == 0].head(10)
    for i, row in safe.iterrows():
        text = str(row[text_col])[:80]
        print(f"  [{i:5d}] {text}")
    
    # Text length distribution
    df['_text_len'] = df[text_col].astype(str).str.len()
    print(f"\nText length statistics:")
    print(f"  Min:    {df['_text_len'].min()}")
    print(f"  Max:    {df['_text_len'].max()}")
    print(f"  Mean:   {df['_text_len'].mean():.1f}")
    print(f"  Median: {df['_text_len'].median():.1f}")
    print(f"  >200:   {(df['_text_len'] > 200).sum()} samples ({(df['_text_len'] > 200).mean()*100:.1f}%)")
    df.drop('_text_len', axis=1, inplace=True)

## 4. VDCNN Inference on Real Data

In [ ]:
# Run VDCNN on example inputs (untrained model — random predictions)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

test_inputs = [
    # SQL Injections (should be detected)
    ("' OR '1'='1", 1),
    ("'; DROP TABLE users--", 1),
    ("' UNION SELECT password,username FROM users--", 1),
    ("' AND SLEEP(5)--", 1),
    ("' AND EXTRACTVALUE(1,CONCAT(0x7e,version()))--", 1),
    ("admin'--", 1),
    ("'; EXEC xp_cmdshell('dir')--", 1),
    ("' OR 1=1; INSERT INTO users VALUES('hacker','pass')--", 1),
    
    # Safe inputs (should NOT be detected)
    ("John O'Brien", 0),
    ("McDonald's restaurant", 0),
    ("normal search query", 0),
    ("test@email.com", 0),
    ("Please select an option", 0),
    ("Drop me a line at john@test.com", 0),
    ("Update your profile information", 0),
    ("It's a beautiful day", 0),
]

print(f"Device: {device}")
print(f"Model: {repr(model)}")
print(f"\nNOTE: Model is UNTRAINED — predictions are random.")
print(f"After training, injection inputs should score >0.5 and safe inputs <0.5.\n")
print(f"{'Input':<55} | {'Expected':>8} | {'Score':>8}")
print("-" * 80)

texts = [t[0] for t in test_inputs]
labels = [t[1] for t in test_inputs]

# Batch tokenize
encoded = tokenizer.encode_batch(texts)
batch_tensor = torch.tensor(encoded, dtype=torch.long).to(device)

with torch.no_grad():
    scores = model(batch_tensor).squeeze().cpu().numpy()

for (text, expected), score in zip(test_inputs, scores):
    label_str = 'INJECT' if expected == 1 else 'SAFE'
    print(f"{text:<55} | {label_str:>8} | {score:>8.4f}")

## 5. Architecture Comparison: Old vs New

In [ ]:
comparison = {
    'Component': [
        'Architecture', 'Year', 'Depth', 'Residual Connections',
        'Filter Progression', 'Pooling', 'FC Layers',
        'Embedding Dim', 'Weight Init', 'Parameters'
    ],
    'Old CharCNN (Kim 2014)': [
        'Sequential Conv', '2014', '3 conv layers', 'None',
        '64 → 128 → 128', 'AdaptiveMaxPool(1)', '128 → 64 → 1',
        '32', 'PyTorch default', '~50K'
    ],
    'New VDCNN (Conneau 2017)': [
        'Deep Residual Conv', '2017', '9-49 conv layers', 'Identity + 1×1 projection',
        '64 → 128 → 256 → 512', 'MaxPool + KMaxPool(8)', '4096 → 2048 → 2048 → 1',
        '16', 'Kaiming (He) normal', f'~{model.count_parameters():,}'
    ]
}

df_comp = pd.DataFrame(comparison)
print(df_comp.to_string(index=False))

## 6. Residual Block Visualization

In [ ]:
# Demonstrate ConvBlock with residual connection
print("ConvBlock — Residual Convolutional Block")
print("=" * 60)

# Same dimension (identity shortcut)
block_identity = ConvBlock(64, 64, kernel_size=3, shortcut=True)
x = torch.randn(2, 64, 100)  # (batch=2, channels=64, length=100)
out = block_identity(x)
print(f"\nIdentity shortcut (same dim):")
print(f"  Input:  {x.shape}  (channels=64)")
print(f"  Output: {out.shape} (channels=64)")
print(f"  Shortcut: identity (no projection needed)")

# Different dimension (1x1 conv projection)
block_proj = ConvBlock(64, 128, kernel_size=3, shortcut=True)
out2 = block_proj(x)
print(f"\nProjection shortcut (dim change):")
print(f"  Input:  {x.shape}  (channels=64)")
print(f"  Output: {out2.shape} (channels=128)")
print(f"  Shortcut: Conv1d(64→128, k=1) + BN")

# Show the residual connection in detail
print(f"\nConvBlock flow:")
print(f"  x ─────────────────────────────────┐")
print(f"  │                                   │ (shortcut)")
print(f"  ├→ Conv1d(k=3) → BN → ReLU         │")
print(f"  ├→ Conv1d(k=3) → BN                 │")
print(f"  └→ + ←──────────────────────────────┘")
print(f"     → ReLU → output")

## 7. KMaxPool1d Demonstration

In [ ]:
# Demonstrate K-Max Pooling
print("KMaxPool1d — Top-K Feature Extraction")
print("=" * 60)

kmax = KMaxPool1d(k=8)

# Create sample feature map
x = torch.randn(1, 4, 25)  # 1 sample, 4 channels, 25 positions
out = kmax(x)

print(f"Input shape:  {x.shape}  (batch=1, channels=4, length=25)")
print(f"Output shape: {out.shape} (batch=1, channels=4, k=8)")

# Show how it works for one channel
channel_0 = x[0, 0].numpy()
top_8 = np.sort(channel_0)[-8:][::-1]
print(f"\nChannel 0 values: {channel_0.round(2).tolist()}")
print(f"Top-8 selected:   {top_8.round(2).tolist()}")
print(f"\n→ Keeps the 8 highest activations from each channel")
print(f"→ Position-invariant: captures WHAT patterns exist, not WHERE")

## 8. Feature Map Flow Through Stages

In [ ]:
# Trace feature map dimensions through the network
print("Feature Map Flow Through VDCNN-9")
print("=" * 70)

x = torch.randint(0, 97, (1, 200))  # 1 sample, 200 chars
model_cpu = CharCNN(vocab_size=97, embed_dim=16, depth=9)
model_cpu.eval()

with torch.no_grad():
    # Manual forward pass with shape tracking
    print(f"{'Layer':<35} | {'Shape':>25} | {'Description'}")
    print("-" * 90)
    
    print(f"{'Input':<35} | {str(x.shape):>25} | Character indices")
    
    h = model_cpu.embedding(x)
    print(f"{'Embedding':<35} | {str(h.shape):>25} | 97 chars → 16-dim vectors")
    
    h = h.permute(0, 2, 1)
    print(f"{'Permute (channels first)':<35} | {str(h.shape):>25} | For Conv1d")
    
    h = model_cpu.initial_conv(h)
    print(f"{'Initial Conv1d(16→64, k=3)':<35} | {str(h.shape):>25} | Initial feature extraction")
    
    for i, (stage, pool) in enumerate(zip(model_cpu.stages, model_cpu.pools)):
        h = stage(h)
        stage_filters = STAGE_FILTERS[i]
        print(f"{'Stage ' + str(i+1) + f' ConvBlock({stage_filters})':<35} | {str(h.shape):>25} | Residual conv block")
        
        h = pool(h)
        pool_type = 'MaxPool(k=3,s=2)' if i < 3 else 'Identity'
        print(f"{'  → ' + pool_type:<35} | {str(h.shape):>25} | Downsample" if i < 3 else f"{'  → ' + pool_type:<35} | {str(h.shape):>25} | No pooling (last stage)")
    
    h = model_cpu.kmax_pool(h)
    print(f"{'KMaxPool(k=8)':<35} | {str(h.shape):>25} | Top-8 activations per channel")
    
    h = h.view(h.size(0), -1)
    print(f"{'Flatten':<35} | {str(h.shape):>25} | 512 × 8 = 4096")
    
    h = model_cpu.classifier(h)
    print(f"{'FC(4096→2048→2048→1)':<35} | {str(h.shape):>25} | Classification")
    
    h = model_cpu.sigmoid(h)
    print(f"{'Sigmoid':<35} | {str(h.shape):>25} | Probability [0, 1]")

## 9. Dataset Interaction — Batch Processing

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SQLiDataset(Dataset):
    """SQL Injection detection dataset."""
    
    def __init__(self, texts, labels, tokenizer, max_length=200):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = float(self.labels[idx])
        tokens = self.tokenizer.encode(text)
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(label, dtype=torch.float32)

# Load dataset and create DataLoader
if 'df' in dir() and df is not None:
    text_col = df.columns[0]
    
    dataset = SQLiDataset(
        texts=df[text_col].values,
        labels=df[label_col].values,
        tokenizer=tokenizer
    )
    
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    # Get one batch
    batch_tokens, batch_labels = next(iter(loader))
    
    print(f"Dataset size: {len(dataset):,}")
    print(f"Batch size: {batch_tokens.shape[0]}")
    print(f"Token tensor shape: {batch_tokens.shape}")
    print(f"Labels shape: {batch_labels.shape}")
    print(f"Labels in batch: {batch_labels.tolist()}")
    
    # Run through model
    model_cpu.eval()
    with torch.no_grad():
        predictions = model_cpu(batch_tokens)
    
    print(f"\nPredictions shape: {predictions.shape}")
    print(f"Predictions (first 10): {predictions.squeeze()[:10].tolist()}")
    print(f"\nNOTE: Predictions are random — model is not trained yet.")
else:
    print("No dataset loaded. Creating synthetic example...")
    texts = ["' OR '1'='1", "normal text", "'; DROP TABLE--", "hello world"]
    labels = [1, 0, 1, 0]
    dataset = SQLiDataset(texts, labels, tokenizer)
    loader = DataLoader(dataset, batch_size=4)
    batch_tokens, batch_labels = next(iter(loader))
    print(f"Synthetic dataset: {len(dataset)} samples")
    print(f"Batch shape: {batch_tokens.shape}")

## 10. Checkpoint Save / Load Test

In [ ]:
# Test save/load compatibility
config = {
    'vocab_size': 97,
    'embed_dim': 16,
    'depth': 9,
    'k_max': 8,
    'fc_dim': 2048,
    'num_classes': 1,
    'shortcut': True,
}

test_path = 'test_vdcnn_checkpoint.pt'
model_cpu.save_checkpoint(test_path, config)
file_size = os.path.getsize(test_path) / 1e6
print(f"Checkpoint saved: {test_path} ({file_size:.1f} MB)")

# Load and verify
loaded_model = CharCNN.load_from_checkpoint(test_path)

test_input = torch.randint(0, 97, (4, 200))
model_cpu.eval()
with torch.no_grad():
    orig_out = model_cpu(test_input)
    loaded_out = loaded_model(test_input)

match = torch.allclose(orig_out, loaded_out, atol=1e-6)
print(f"Outputs match after load: {match}")
print(f"Loaded model: {repr(loaded_model)}")

# Cleanup
os.remove(test_path)
print(f"\n✓ Checkpoint save/load working correctly!")

## 11. CUDA Performance Preview

In [ ]:
import time

# Benchmark inference speed
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_bench = CharCNN(vocab_size=97, embed_dim=16, depth=9).to(device)
model_bench.eval()

batch_sizes = [32, 128, 512, 2048]
n_samples = 10000

print(f"Inference Benchmark on {device.upper()}")
print(f"{'Batch Size':>12} | {'Throughput':>15} | {'Latency/sample':>15}")
print("-" * 50)

for bs in batch_sizes:
    if bs > n_samples:
        continue
    
    # Warm up
    dummy = torch.randint(0, 97, (bs, 200)).to(device)
    with torch.no_grad():
        _ = model_bench(dummy)
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    n_batches = n_samples // bs
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_batches):
            _ = model_bench(dummy)
    
    if device == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.perf_counter() - start
    total_processed = n_batches * bs
    throughput = total_processed / elapsed
    latency = elapsed / total_processed * 1000  # ms
    
    print(f"{bs:>12} | {throughput:>12,.0f}/sec | {latency:>12.4f} ms")

print(f"\nDevice: {device}")
if device == 'cuda':
    mem = torch.cuda.max_memory_allocated() / 1e6
    print(f"Peak GPU memory: {mem:.1f} MB")

## Summary

**VDCNN model is ready for training.**

Next steps:
1. Update `training/train_cnn.py` with new hyperparameters (SGD, lr=0.01, grad_clip=7.0)
2. Train on dataset with CUDA
3. Generate 100k+ test data
4. Run CUDA benchmark
5. Stress test